In [28]:
import pandas as pd
import numpy as np

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

RANDOM_STATE = 42

train_df = pd.read_csv("../data/processed/train.csv")
test_df = pd.read_csv("../data/processed/test.csv")

TARGET = "blackout"

X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

X_test = test_df.drop(columns=[TARGET])
y_test = test_df[TARGET]

assert list(X_train.columns) == list(X_test.columns)
assert TARGET not in X_train.columns
assert len(X_train) == 16018
assert len(X_test) == 3982

print("TRAINING SET")
print("Shape:", X_train.shape)
print("Class distribution:")
print(y_train.value_counts().sort_index())

print("\nTEST SET")
print("Shape:", X_test.shape)
print("Class distribution:")
print(y_test.value_counts().sort_index())

print("\nStratifiedGroupKFold available:", StratifiedGroupKFold)
print("\nData loading completed successfully.")


TRAINING SET
Shape: (16018, 224)
Class distribution:
0    12778
1     3240
Name: blackout, dtype: int64

TEST SET
Shape: (3982, 224)
Class distribution:
0    3190
1     792
Name: blackout, dtype: int64

StratifiedGroupKFold available: <class 'sklearn.model_selection._split.StratifiedGroupKFold'>

Data loading completed successfully.


In [29]:
# Create group IDs from all 224 input features.
# Identical feature vectors receive the same group ID.

groups = pd.util.hash_pandas_object(
    X_train,
    index=False
).to_numpy()

# Check that the hashes represent distinct feature vectors.
unique_feature_vectors = len(X_train.drop_duplicates())
unique_groups = len(np.unique(groups))

assert unique_feature_vectors == unique_groups, (
    "Hash collision detected or group construction is inconsistent."
)

# Check that each group contains only one target label.
group_label_counts = (
    pd.DataFrame({
        "group": groups,
        "target": y_train.to_numpy()
    })
    .groupby("group")["target"]
    .nunique()
)

assert group_label_counts.max() == 1, (
    "A group contains different target labels."
)

# Configure 5-fold stratified group cross-validation.
cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("Total training records:", len(X_train))
print("Unique feature-vector groups:", unique_groups)
print("Duplicate records grouped:", len(X_train) - unique_groups)
print("Number of folds:", cv.get_n_splits())

# Verify that no group is shared between training and validation folds.
for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_train, y_train, groups),
    start=1
):
    train_groups = set(groups[train_idx])
    val_groups = set(groups[val_idx])

    assert train_groups.isdisjoint(val_groups)

    print(f"\nFold {fold}")
    print("Training records:", len(train_idx))
    print("Validation records:", len(val_idx))
    print("Training blackout records:", int(y_train.iloc[train_idx].sum()))
    print("Validation blackout records:", int(y_train.iloc[val_idx].sum()))
    print("Shared groups:", len(train_groups.intersection(val_groups)))

print("\nGrouped cross-validation checks passed.")


Total training records: 16018
Unique feature-vector groups: 14863
Duplicate records grouped: 1155
Number of folds: 5

Fold 1
Training records: 12811
Validation records: 3207
Training blackout records: 2593
Validation blackout records: 647
Shared groups: 0

Fold 2
Training records: 12820
Validation records: 3198
Training blackout records: 2609
Validation blackout records: 631
Shared groups: 0

Fold 3
Training records: 12809
Validation records: 3209
Training blackout records: 2593
Validation blackout records: 647
Shared groups: 0

Fold 4
Training records: 12814
Validation records: 3204
Training blackout records: 2597
Validation blackout records: 643
Shared groups: 0

Fold 5
Training records: 12818
Validation records: 3200
Training blackout records: 2568
Validation blackout records: 672
Shared groups: 0

Grouped cross-validation checks passed.


In [30]:
models = {
    "Baseline Decision Tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ),
    "Controlled DT 1": DecisionTreeClassifier(
        max_depth=8,
        min_samples_leaf=5,
        random_state=RANDOM_STATE
    ),
    "Controlled DT 2": DecisionTreeClassifier(
        max_depth=10,
        min_samples_leaf=10,
        random_state=RANDOM_STATE
    )
}

print("Models defined:")
for name, model in models.items():
    print(f"{name}: {model.get_params()}")


Models defined:
Baseline Decision Tree: {'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': None, 'max_features': None, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'random_state': 42, 'splitter': 'best'}
Controlled DT 1: {'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': 8, 'max_features': None, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 5, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'random_state': 42, 'splitter': 'best'}
Controlled DT 2: {'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': 10, 'max_features': None, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 10, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'random_state': 42, 'splitter': 'best'}


In [31]:
from sklearn.base import clone

oof_results = {}

for model_name, model in models.items():
    print(f"\nRunning: {model_name}")

    oof_predictions = np.zeros(len(X_train), dtype=int)
    oof_probabilities = np.zeros(len(X_train), dtype=float)

    for fold, (train_idx, val_idx) in enumerate(
        cv.split(X_train, y_train, groups),
        start=1
    ):
        X_fold_train = X_train.iloc[train_idx]
        y_fold_train = y_train.iloc[train_idx]

        X_fold_val = X_train.iloc[val_idx]

        fold_model = clone(model)
        fold_model.fit(X_fold_train, y_fold_train)

        oof_predictions[val_idx] = fold_model.predict(X_fold_val)

        positive_class_index = list(fold_model.classes_).index(1)
        oof_probabilities[val_idx] = (
            fold_model.predict_proba(X_fold_val)[:, positive_class_index]
        )

        print(f"  Fold {fold}: completed")

    oof_results[model_name] = {
        "predictions": oof_predictions,
        "probabilities": oof_probabilities
    }

    print(f"  OOF predictions generated: {len(oof_predictions)}")

print("\nAll OOF predictions generated successfully.")



Running: Baseline Decision Tree
  Fold 1: completed
  Fold 2: completed
  Fold 3: completed
  Fold 4: completed
  Fold 5: completed
  OOF predictions generated: 16018

Running: Controlled DT 1
  Fold 1: completed
  Fold 2: completed
  Fold 3: completed
  Fold 4: completed
  Fold 5: completed
  OOF predictions generated: 16018

Running: Controlled DT 2
  Fold 1: completed
  Fold 2: completed
  Fold 3: completed
  Fold 4: completed
  Fold 5: completed
  OOF predictions generated: 16018

All OOF predictions generated successfully.


In [32]:
comparison_results = []

for model_name, result in oof_results.items():
    y_pred = result["predictions"]
    y_prob = result["probabilities"]

    comparison_results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_train, y_pred),
        "Precision": precision_score(y_train, y_pred, zero_division=0),
        "Recall": recall_score(y_train, y_pred, zero_division=0),
        "F1-score": f1_score(y_train, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_train, y_prob)
    })

comparison_df = pd.DataFrame(comparison_results)

print("========== GROUPED CROSS-VALIDATION RESULTS ==========")
display(
    comparison_df.style.format({
        "Accuracy": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1-score": "{:.4f}",
        "ROC-AUC": "{:.4f}"
    })
)


========== GROUPED CROSS-VALIDATION RESULTS ==========


,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Baseline Decision Tree,0.9871,0.9711,0.9648,0.9680,0.9788
1,Controlled DT 1,0.9148,0.7522,0.8636,0.8040,0.9597
2,Controlled DT 2,0.9293,0.7953,0.8756,0.8336,0.9690


In [33]:
oof_predictions = {}
oof_probabilities = {}

for name, model in models.items():
    print(f"\nRunning: {name}")

    predictions = np.zeros(len(X_train), dtype=int)
    probabilities = np.zeros(len(X_train), dtype=float)

    for fold, (train_idx, val_idx) in enumerate(
        cv.split(X_train, y_train, groups),
        start=1
    ):
        fold_model = DecisionTreeClassifier(**model.get_params())

        fold_model.fit(
            X_train.iloc[train_idx],
            y_train.iloc[train_idx]
        )

        predictions[val_idx] = fold_model.predict(
            X_train.iloc[val_idx]
        )

        probabilities[val_idx] = fold_model.predict_proba(
            X_train.iloc[val_idx]
        )[:, 1]

        print(f"  Fold {fold}: completed")

    oof_predictions[name] = predictions
    oof_probabilities[name] = probabilities

    print("  OOF predictions:", len(predictions))
    print("  OOF probabilities:", len(probabilities))

print("\nAll model-specific OOF results generated.")



Running: Baseline Decision Tree
  Fold 1: completed
  Fold 2: completed
  Fold 3: completed
  Fold 4: completed
  Fold 5: completed
  OOF predictions: 16018
  OOF probabilities: 16018

Running: Controlled DT 1
  Fold 1: completed
  Fold 2: completed
  Fold 3: completed
  Fold 4: completed
  Fold 5: completed
  OOF predictions: 16018
  OOF probabilities: 16018

Running: Controlled DT 2
  Fold 1: completed
  Fold 2: completed
  Fold 3: completed
  Fold 4: completed
  Fold 5: completed
  OOF predictions: 16018
  OOF probabilities: 16018

All model-specific OOF results generated.


In [34]:
cv_results = []

for name in models:
    y_pred = oof_predictions[name]
    y_prob = oof_probabilities[name]

    cv_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_train, y_pred),
        "Precision": precision_score(y_train, y_pred, zero_division=0),
        "Recall": recall_score(y_train, y_pred, zero_division=0),
        "F1-score": f1_score(y_train, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_train, y_prob)
    })

cv_results_df = pd.DataFrame(cv_results)

print("========== GROUPED CROSS-VALIDATION RESULTS ==========")
print(cv_results_df.round(4).to_string(index=False))

best_model_name = cv_results_df.loc[
    cv_results_df["F1-score"].idxmax(), "Model"
]

print("\nSelected model:", best_model_name)


========== GROUPED CROSS-VALIDATION RESULTS ==========
                 Model  Accuracy  Precision  Recall  F1-score  ROC-AUC
Baseline Decision Tree    0.9871     0.9711  0.9648    0.9680   0.9788
       Controlled DT 1    0.9148     0.7522  0.8636    0.8040   0.9597
       Controlled DT 2    0.9293     0.7953  0.8756    0.8336   0.9690

Selected model: Baseline Decision Tree


In [35]:
best_oof_probabilities = oof_probabilities[best_model_name]

threshold_results = []

thresholds = np.arange(0.1, 1.0, 0.1)

for threshold in thresholds:
    y_pred_threshold = (
        best_oof_probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": round(threshold, 2),
        "Accuracy": accuracy_score(y_train, y_pred_threshold),
        "Precision": precision_score(
            y_train, y_pred_threshold, zero_division=0
        ),
        "Recall": recall_score(
            y_train, y_pred_threshold, zero_division=0
        ),
        "F1-score": f1_score(
            y_train, y_pred_threshold, zero_division=0
        )
    })

threshold_df = pd.DataFrame(threshold_results)

print("========== OOF THRESHOLD ANALYSIS ==========")
print(threshold_df.round(4).to_string(index=False))

best_threshold = threshold_df.loc[
    threshold_df["F1-score"].idxmax(),
    "Threshold"
]

print("\nSelected threshold:", best_threshold)


========== OOF THRESHOLD ANALYSIS ==========
 Threshold  Accuracy  Precision  Recall  F1-score
       0.1    0.9871     0.9711  0.9648     0.968
       0.2    0.9871     0.9711  0.9648     0.968
       0.3    0.9871     0.9711  0.9648     0.968
       0.4    0.9871     0.9711  0.9648     0.968
       0.5    0.9871     0.9711  0.9648     0.968
       0.6    0.9871     0.9711  0.9648     0.968
       0.7    0.9871     0.9711  0.9648     0.968
       0.8    0.9871     0.9711  0.9648     0.968
       0.9    0.9871     0.9711  0.9648     0.968

Selected threshold: 0.1


In [36]:
probability_values = np.unique(best_oof_probabilities)

print("Number of unique probability values:", len(probability_values))
print("\nFirst 30 unique probability values:")
print(probability_values[:30])

print("\nProbability distribution:")
print(
    pd.Series(best_oof_probabilities)
    .value_counts()
    .sort_index()
    .to_string()
)


Number of unique probability values: 2

First 30 unique probability values:
[0. 1.]

Probability distribution:
0.0    12799
1.0     3219


### Threshold Analysis

The threshold analysis was performed using out-of-fold predictions from the selected Baseline Decision Tree. All tested thresholds, ranging from 0.1 to 0.9, produced identical classification metrics.

This behaviour is explained by the model's discrete probability outputs: only two probability values were observed (0.0 and 1.0). Consequently, changing the classification threshold within the tested range did not affect the predicted classes.

The default threshold of 0.5 was therefore retained. These probability outputs should not be interpreted as calibrated estimates of real-world blackout risk.

In [37]:
# Refit the selected model using the complete training set.

final_model = DecisionTreeClassifier(
    **models[best_model_name].get_params()
)

final_model.fit(X_train, y_train)

print("Final model:", best_model_name)
print("Training records:", len(X_train))
print("Tree depth:", final_model.get_depth())
print("Number of leaves:", final_model.get_n_leaves())
print("Number of nodes:", final_model.tree_.node_count)
print("Final model fitted successfully.")


Final model: Baseline Decision Tree
Training records: 16018
Tree depth: 27
Number of leaves: 232
Number of nodes: 463
Final model fitted successfully.


In [38]:
# Final evaluation on the untouched test set.

y_test_pred = final_model.predict(X_test)
y_test_prob = final_model.predict_proba(X_test)[:, 1]

test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred, zero_division=0)
test_recall = recall_score(y_test, y_test_pred, zero_division=0)
test_f1 = f1_score(y_test, y_test_pred, zero_division=0)
test_roc_auc = roc_auc_score(y_test, y_test_prob)

test_cm = confusion_matrix(y_test, y_test_pred)

print("========== FINAL TEST RESULTS ==========")
print("Test records:", len(X_test))
print("Accuracy:", round(test_accuracy, 4))
print("Precision:", round(test_precision, 4))
print("Recall:", round(test_recall, 4))
print("F1-score:", round(test_f1, 4))
print("ROC-AUC:", round(test_roc_auc, 4))

print("\nConfusion Matrix:")
print(test_cm)

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_test_pred,
    target_names=["Normal (0)", "Blackout (1)"],
    digits=4
))

========== FINAL TEST RESULTS ==========
Test records: 3982
Accuracy: 0.9922
Precision: 0.986
Recall: 0.9747
F1-score: 0.9803
ROC-AUC: 0.9856

Confusion Matrix:
[[3179   11]
 [  20  772]]

Classification Report:
              precision    recall  f1-score   support

  Normal (0)     0.9937    0.9966    0.9951      3190
Blackout (1)     0.9860    0.9747    0.9803       792

    accuracy                         0.9922      3982
   macro avg     0.9898    0.9856    0.9877      3982
weighted avg     0.9922    0.9922    0.9922      3982



In [39]:
# Identify misclassified test records.

y_test_array = y_test.to_numpy()
error_mask = y_test_array != y_test_pred

error_positions = np.where(error_mask)[0]

test_error_df = pd.DataFrame({
    "test_row_index": X_test.index[error_positions],
    "actual": y_test_array[error_positions],
    "predicted": y_test_pred[error_positions],
    "predicted_probability": y_test_prob[error_positions],
})

test_error_df["error_type"] = np.where(
    test_error_df["actual"] == 1,
    "False Negative",
    "False Positive"
)

print("========== ERROR ANALYSIS ==========")
print("Total errors:", len(test_error_df))
print("\nError types:")
print(test_error_df["error_type"].value_counts())

print("\nFalse Positives:")
print(
    test_error_df[
        test_error_df["error_type"] == "False Positive"
    ].to_string(index=False)
)

print("\nFalse Negatives:")
print(
    test_error_df[
        test_error_df["error_type"] == "False Negative"
    ].to_string(index=False)
)

# Save the error records for the final report.
test_error_df.to_csv(
    "../reports/decision_tree_test_errors.csv",
    index=False
)

print("\nError analysis file saved.")

========== ERROR ANALYSIS ==========
Total errors: 31

Error types:
False Negative    20
False Positive    11
Name: error_type, dtype: int64

False Positives:
 test_row_index  actual  predicted  predicted_probability     error_type
            266       0          1                    1.0 False Positive
            360       0          1                    1.0 False Positive
           1501       0          1                    1.0 False Positive
           2003       0          1                    1.0 False Positive
           2041       0          1                    1.0 False Positive
           2330       0          1                    1.0 False Positive
           2787       0          1                    1.0 False Positive
           2879       0          1                    1.0 False Positive
           2952       0          1                    1.0 False Positive
           3424       0          1                    1.0 False Positive
           3918       0          1    

In [40]:
# Compare misclassified and correctly classified records.
# All 224 input features are numerical.

test_analysis_df = X_test.copy()
test_analysis_df["actual"] = y_test.to_numpy()
test_analysis_df["predicted"] = y_test_pred

test_analysis_df["prediction_group"] = np.select(
    [
        (test_analysis_df["actual"] == 1) &
        (test_analysis_df["predicted"] == 1),

        (test_analysis_df["actual"] == 1) &
        (test_analysis_df["predicted"] == 0),

        (test_analysis_df["actual"] == 0) &
        (test_analysis_df["predicted"] == 0),

        (test_analysis_df["actual"] == 0) &
        (test_analysis_df["predicted"] == 1)
    ],
    [
        "True Positive",
        "False Negative",
        "True Negative",
        "False Positive"
    ]
)

print("========== PREDICTION GROUP COUNTS ==========")
print(test_analysis_df["prediction_group"].value_counts())

# Compare feature means between false negatives and true positives.
blackout_comparison = test_analysis_df[
    test_analysis_df["actual"] == 1
].groupby("prediction_group")[X_test.columns].mean().T

# Compare feature means between false positives and true negatives.
normal_comparison = test_analysis_df[
    test_analysis_df["actual"] == 0
].groupby("prediction_group")[X_test.columns].mean().T

# Calculate standardized mean differences.
def calculate_smd(group_a, group_b):
    pooled_std = np.sqrt(
        (group_a.var() + group_b.var()) / 2
    )
    smd = (group_a.mean() - group_b.mean()) / pooled_std.replace(0, np.nan)
    return smd.replace([np.inf, -np.inf], np.nan).dropna()

fn_vs_tp = calculate_smd(
    test_analysis_df.loc[
        test_analysis_df["prediction_group"] == "False Negative",
        X_test.columns
    ],
    test_analysis_df.loc[
        test_analysis_df["prediction_group"] == "True Positive",
        X_test.columns
    ]
)

fp_vs_tn = calculate_smd(
    test_analysis_df.loc[
        test_analysis_df["prediction_group"] == "False Positive",
        X_test.columns
    ],
    test_analysis_df.loc[
        test_analysis_df["prediction_group"] == "True Negative",
        X_test.columns
    ]
)

print("\n========== FALSE NEGATIVE VS TRUE POSITIVE ==========")
print("Top 10 feature differences:")
print(
    fn_vs_tp.abs()
    .sort_values(ascending=False)
    .head(10)
    .to_string()
)

print("\n========== FALSE POSITIVE VS TRUE NEGATIVE ==========")
print("Top 10 feature differences:")
print(
    fp_vs_tn.abs()
    .sort_values(ascending=False)
    .head(10)
    .to_string()
)


========== PREDICTION GROUP COUNTS ==========
True Negative     3179
True Positive      772
False Negative      20
False Positive      11
Name: prediction_group, dtype: int64

========== FALSE NEGATIVE VS TRUE POSITIVE ==========
Top 10 feature differences:
line_9_reactive_power_flow     0.937373
line_7_reactive_power_flow     0.808485
line_4_reactive_power_flow     0.796484
bus_20_net_apparent_power      0.788067
bus_6_net_apparent_power       0.774927
line_8_reactive_power_flow     0.762837
line_5_reactive_power_flow     0.757479
bus_4_net_apparent_power       0.748665
line_3_reactive_power_flow     0.747141
line_30_reactive_power_flow    0.746579

========== FALSE POSITIVE VS TRUE NEGATIVE ==========
Top 10 feature differences:
bus_3_voltage_magnitude        1.108691
line_24_reactive_power_flow    1.098624
bus_14_voltage_magnitude       1.094376
line_14_reactive_power_flow    1.093723
bus_23_voltage_magnitude       1.088831
bus_20_voltage_magnitude       1.086967
bus_6_voltage_magni

In [41]:
# Save the final Decision Tree experiment summary.

final_results = cv_results_df.copy()

final_results["Evaluation"] = "Grouped 5-fold OOF"
final_results["Selected"] = final_results["Model"] == best_model_name

test_summary = pd.DataFrame([{
    "Model": best_model_name,
    "Accuracy": test_accuracy,
    "Precision": test_precision,
    "Recall": test_recall,
    "F1-score": test_f1,
    "ROC-AUC": test_roc_auc,
    "Evaluation": "Independent test",
    "Selected": True
}])

final_results = pd.concat(
    [final_results, test_summary],
    ignore_index=True
)

final_results.to_csv(
    "../reports/decision_tree_final_results.csv",
    index=False
)

# Save the threshold analysis.
threshold_df.to_csv(
    "../reports/decision_tree_threshold_analysis.csv",
    index=False
)

# Save feature difference analysis.
feature_error_summary = pd.DataFrame({
    "False_Negative_vs_True_Positive_SMD": fn_vs_tp,
    "False_Positive_vs_True_Negative_SMD": fp_vs_tn
})

feature_error_summary.to_csv(
    "../reports/decision_tree_error_feature_analysis.csv"
)

print("Saved experiment files:")
print("../reports/decision_tree_final_results.csv")
print("../reports/decision_tree_threshold_analysis.csv")
print("../reports/decision_tree_error_feature_analysis.csv")
print("../reports/decision_tree_test_errors.csv")


Saved experiment files:
../reports/decision_tree_final_results.csv
../reports/decision_tree_threshold_analysis.csv
../reports/decision_tree_error_feature_analysis.csv
../reports/decision_tree_test_errors.csv
